# Final Project Solutions

## 1. Scope the Project

Our plan is to use a supervised learning technique to predict which customers are most likely to cancel their subscription using **the past three months of customer data which includes subscription and listening history**.

## 2. Gather Data

Read the following files into Python:
* Customer data: *maven_music_customers.csv*
* Listing history: *maven_music_listening_history.xlsx*

In [ ]:
# Read in the customer data
import pandas as pd

customers = pd.read_csv('../Data/maven_music_customers.csv')
customers.head()

In [ ]:
# Read in the listening history
listening_history = pd.read_excel('../Data/maven_music_listening_history.xlsx')
listening_history.head()

In [ ]:
# Hint: Where might you find listening history data beyond the ID's?
## Check the other sheets in the Excel spreadsheet

In [ ]:
# Read in the audio data
audio = pd.read_excel('../Data/maven_music_listening_history.xlsx', sheet_name=1)
audio.head()

In [ ]:
# Read in the session data
sessions = pd.read_excel('../Data/maven_music_listening_history.xlsx', sheet_name=2)
sessions.head()

## 3. Clean Data

### a. Convert Data Types

Check the data types of the data in the tables and convert to numeric and datetime values as necessary.

In [ ]:
# Check the customers data types -- lots of objects to deal with!
customers.dtypes

In [ ]:
# Check the listening history data types -- looks good
listening_history.dtypes

In [ ]:
# Check the audio data types -- looks good
audio.dtypes

In [ ]:
# Check the session data types -- looks good
sessions.dtypes

In [ ]:
customers.head()

In [ ]:
# Convert objects to numeric and datetime fields
customers['Member Since'] = pd.to_datetime(customers['Member Since'])
customers['Subscription Rate'] = pd.to_numeric(customers['Subscription Rate'].str.replace('$', ''))
customers['Cancellation Date'] = pd.to_datetime(customers['Cancellation Date'])

In [ ]:
# New data types look good
customers.dtypes

### b. Resolve Data Issues

Check for missing data, inconsistent text and typos, duplicate data and outliers.

#### i. Missing Data

In [ ]:
# Find the NaN values in the data
customers.info()

In [ ]:
# No NaNs in listening history
listening_history.info()

In [ ]:
# No NaNs in audio
audio.info()

In [ ]:
# No NaNs in sessions
sessions.info()

In [ ]:
# The customers dataframe has null values in the fields Subscription Plan, Discount? and Cancellation Date
customers.head()

In [ ]:
# Look into Subscription Plan -- all NaN Subscription Plans are $2.99
customers[customers['Subscription Plan'].isna()]

In [ ]:
# Check the unique Subscription Rates
customers[['Subscription Rate', 'Subscription Plan']].drop_duplicates()

In [ ]:
# It looks like the $2.99 rate is for Basic plan, so fill missing Subscription Plan values with 'Basic'
customers['Subscription Plan'] = customers['Subscription Plan'].fillna('Basic (Ads)')
customers.head()

In [ ]:
# Look into Discount?
customers[['Customer ID', 'Discount?']].tail()

In [ ]:
# Let's view it another way -- NaN seems to mean No
customers['Discount?'].value_counts()

In [ ]:
# Let's change to numeric to make our life easier later
import numpy as np

customers['Discount?'] = np.where(customers['Discount?']=='Yes', 1, 0)
customers.head()

In [ ]:
# Look into Cancellation Date -- NaN seems to mean not cancelled yet -- let's leave it as is

#### ii. Inconsistent Text & Typos

In [ ]:
# Look at customers -- the subscription rate looks really high
customers.describe()

In [ ]:
# Look into the 99.99 -- looks like a typo
customers[customers['Subscription Rate'] > 7.99]

In [ ]:
# Fix the 99.99 typo
customers.iloc[15, 5] = 9.99

In [ ]:
# Check the date range of customers
customers['Member Since'].max()

In [ ]:
# Look at listening history
listening_history.describe()

In [ ]:
# View the data
listening_history.head()

In [ ]:
# Look at Audio Type values -- looks good
listening_history['Audio Type'].value_counts()

In [ ]:
# Look at the audio data
audio.head()

In [ ]:
# Look into Genre
audio.Genre.value_counts()

In [ ]:
# Pop and Pop Music should be mapped to the same value
audio.Genre = np.where(audio.Genre == 'Pop Music', 'Pop', audio.Genre)
audio

In [ ]:
# Look at log in time range
sessions['Session Log In Time'].max()

#### iii. Duplicate Rows

In [ ]:
# There seem to be no duplicate rows

In [ ]:
customers[customers.duplicated()]

In [ ]:
listening_history[listening_history.duplicated()]

In [ ]:
audio[audio.duplicated()]

In [ ]:
sessions[sessions.duplicated()]

#### iv. Outliers

In [ ]:
# There seem to be no more obvious outliers

In [ ]:
customers.describe()

In [ ]:
listening_history.describe()

In [ ]:
audio.describe()

In [ ]:
sessions.describe()

### c. Create New Columns

Create two new columns that will be useful for EDA and modeling:
* Cancelled: whether a customer cancelled or not
* Email: Remove the "Email:" from the email addresses

In [ ]:
customers.head()

In [ ]:
# Create a 'Cancelled' column
customers['Cancelled'] = np.where(customers['Cancellation Date'].notna(), 1, 0)
customers.head()

In [ ]:
# Create an updated Email column without the Email: portion
customers['Email'] = customers.Email.str[6:]
customers.head()

## 4. EDA

Try to better understand the customers who cancelled:
* How long were they members before they cancelled?
* What percentage of customers who cancelled had a discount vs customers who didn't cancel?

In [ ]:
customers.head()

In [ ]:
# View the customers who cancelled
customers[customers['Cancellation Date'].notna()].head()

In [ ]:
# How long were customers members before they cancelled?
## About 1.5 months... but that might just because we have 3 months of data
(customers['Cancellation Date'] - customers['Member Since']).mean()

In [ ]:
# Calculate the cancellation rate for those who had a discount
discount_yes = customers[customers['Discount?']==1]
discount_yes

In [ ]:
# Cancellation rate for those who had a discount
discount_yes.Cancelled.sum() / discount_yes.Cancelled.count()

In [ ]:
# Calculate the cancellation rate for those who did not have a discount
discount_no = customers[customers['Discount?']==0]
discount_no.head()

In [ ]:
# Cancellation rate for those who did not have a discount
discount_no.Cancelled.sum() / discount_no.Cancelled.count()

In [ ]:
# Visualize the cancellation rate for those with a discount vs those without a discount -- this is a good insight
pd.DataFrame([['Had Discount', 0.8571428571428571],
             ['Did Not Have Discount', 0.30434782608695654]],
             columns=['Customer Type', 'Cancellation Rate']).plot.barh(x='Customer Type', y='Cancellation Rate', );

Better understand the customers' listening histories:
* Join together the listening history and audio tables
* How many listening sessions did each customer have in the past 3 months?
* What were the most popular genres that customers listened to?

In [ ]:
customers.head()

In [ ]:
listening_history.head()

In [ ]:
audio.head()

In [ ]:
sessions.head()

In [ ]:
# Split the ID in the audio data so the column can be joined with other tables
audio_clean = pd.DataFrame(audio.ID.str.split('-').to_list()).rename(columns={0:'Type', 1:'Audio ID'})
audio_clean.head()

In [ ]:
# Add the new fields to the original audio table
audio_all = pd.concat([audio_clean, audio], axis=1)
audio_all.head()

In [ ]:
listening_history.head()

In [ ]:
# If you try to join on the Audio ID, you will get an error
# df = listening_history.merge(audio_all, how='left', on='Audio ID')

In [ ]:
# Hint: Check the data type of Audio ID in the audio table
audio_all.dtypes

In [ ]:
# Change Audio ID to an int type instead of an object
audio_all['Audio ID'] = audio_all['Audio ID'].astype('int')
audio_all.head()

In [ ]:
# Try the merge again
df = listening_history.merge(audio_all, how='left', on='Audio ID')
df

In [ ]:
df.groupby('Customer ID')['Session ID'].nunique()

In [ ]:
# The number of listening sessions that each customer had in the past 3 months
df.groupby('Customer ID')['Session ID'].nunique().plot.hist();

In [ ]:
# The most popular genres that customers listened to
df.Genre.value_counts()

## 5. Prep for Modeling

Create a DataFrame that is ready for modeling with each row representing a customer and the following numeric, non-null columns:
* Customer ID
* Whether a customer cancelled or not
* Whether a customer received a discount or not
* The number of listening sessions
* Percent of listening history consisting of Pop
* Percent of listening history consisting of Podcasts

In [ ]:
# Create a dataframe ready for modeling
model_df = customers[['Customer ID', 'Cancelled', 'Discount?']]
model_df.head()

In [ ]:
df.head()

In [ ]:
df.groupby('Customer ID')['Session ID'].nunique()

In [ ]:
# Calculate the number of listening sessions for each customers
number_of_sessions = df.groupby('Customer ID')['Session ID'].nunique().rename('Number of Sessions').to_frame().reset_index()
number_of_sessions.head()

In [ ]:
# Add it to the modeling dataframe
model_df = model_df.merge(number_of_sessions, how='left', on='Customer ID')
model_df.head()

In [ ]:
df.Genre

In [ ]:
# Calculate dummy variables for each genre
pd.get_dummies(df.Genre)

In [ ]:
# Combine it with the customer ID
pd.concat([df['Customer ID'], pd.get_dummies(df.Genre)], axis=1)

In [ ]:
# Group it by customer
genres = pd.concat([df['Customer ID'], pd.get_dummies(df.Genre)], axis=1).groupby('Customer ID').sum().reset_index()
genres.head()

In [ ]:
listening_history.head()

In [ ]:
# Add a column for total songs / podcasts listened to
total_audio = listening_history.groupby('Customer ID')['Audio ID'].count().rename('Total Audio').to_frame().reset_index()
total_audio.head()

In [ ]:
# Create a master audio table to calculate percentages
df_audio = genres.merge(total_audio, how='left', on='Customer ID')
df_audio

In [ ]:
# Percent pop
model_df['Percent Pop'] = df_audio.Pop / df_audio['Total Audio'] * 100
model_df.head()

In [ ]:
# Percent podcasts
model_df['Percent Podcasts'] = ((df_audio['Comedy'] + df_audio['True Crime']) / df_audio['Total Audio']) * 100
model_df.head()

In [ ]:
model_df

Visualize the relationships in the modeling DataFrame using a pair plot:
* What are some of your observations?
* What variables might do a good job predicting customer cancellation?

In [ ]:
import seaborn as sns

In [ ]:
sns.pairplot(model_df);

In [ ]:
# Look at the correlations
model_df.corr()

In [ ]:
# Observations
## A discount is correlated with a cancellation
## The more listening sessions, the fewer cancellations
## The more pop music, the more cancellations
## Podcast listening history seems unrelated to cancellations